# 🇮🇳 Hindi Corpus Collection (Model H)

**Phase 1 — Data Collection Pipeline**

Strategy: PDF/ASR first (zero wiki overlap) → Wiki dump (manual) → Web last

**Runtime**: CPU | **Time**: ~5 hours

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/24KaratAu/hindi'
for d in ['raw/wiki_dump', 'raw/hf_automated', 'cleaned/manual', 'cleaned/automated', 'cleaned/merged', 'splits', 'tokenizer']:
    os.makedirs(f'{DRIVE_BASE}/{d}', exist_ok=True)
print('✅ Drive directories created')

In [ ]:
!pip install -q wikiextractor datasets sentencepiece
print('✅ Dependencies installed')

In [ ]:
import json, hashlib, re, random, bz2, os
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

LANGUAGE = 'hindi'
WORK_DIR = '/content/hindi_corpus'
os.makedirs(WORK_DIR, exist_ok=True)

MIN_CHARS = 100
MIN_DEV_RATIO = 0.5
SHARD_SIZE = 10000

def devanagari_ratio(text):
    if not text: return 0.0
    return sum(1 for c in text if '\u0900' <= c <= '\u097F') / len(text)

def clean_web_text(text):
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    text = re.sub(r'&#\d+;', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def text_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

all_hashes = set()

print(f'✅ Ready | Work: {WORK_DIR} | Drive: {DRIVE_BASE}')

---
## Step 1: Sangraha PDF + ASR (automated, zero wiki overlap)

In [ ]:
from datasets import load_dataset

NONWEB_DIR = f'{WORK_DIR}/automated_nonweb'
os.makedirs(NONWEB_DIR, exist_ok=True)

nonweb_docs = []
nonweb_shard_idx = 0
nonweb_total_chars = 0
nonweb_total_docs = 0
type_counts = Counter()
lang_hits = 0
total_streamed = 0

def flush_nonweb():
    global nonweb_docs, nonweb_shard_idx
    if not nonweb_docs: return
    sf = f'{NONWEB_DIR}/nonweb_shard_{nonweb_shard_idx:04d}.jsonl'
    with open(sf, 'w', encoding='utf-8') as f:
        for d in nonweb_docs:
            f.write(json.dumps(d, ensure_ascii=False) + '\n')
    print(f'  ✅ {sf}: {len(nonweb_docs):,} docs')
    nonweb_docs = []
    nonweb_shard_idx += 1

print('Streaming ai4bharat/sangraha verified — collecting PDF + ASR only...')
print('(Web type is skipped now, collected later in Step 3)\n')

try:
    ds = load_dataset('ai4bharat/sangraha', 'verified', split='train',
                      streaming=True, trust_remote_code=True)
    for ex in ds:
        total_streamed += 1
        lang = ex.get('language', '') or ex.get('lang', '')
        if lang not in ('hin', 'hi', 'Hindi', 'hindi'):
            continue
        lang_hits += 1
        doc_type = (ex.get('type', '') or '').strip().lower()
        type_counts[doc_type] += 1
        
        if 'web' in doc_type:
            continue
        
        text = ex.get('text', '')
        if not text: continue
        cleaned = clean_web_text(text)
        if len(cleaned) < MIN_CHARS: continue
        if devanagari_ratio(cleaned) < MIN_DEV_RATIO: continue
        
        h = text_hash(cleaned)
        if h in all_hashes: continue
        all_hashes.add(h)
        
        nonweb_docs.append({
            'doc_id': hashlib.md5(h.encode()).hexdigest()[:16],
            'source': 'sangraha', 'source_type': f'hf_sangraha_{doc_type}',
            'collection_type': 'automated', 'language': LANGUAGE,
            'text': cleaned, 'clean_text_characters': len(cleaned),
            'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
            'text_sha256': h, 'sangraha_type': doc_type,
        })
        nonweb_total_chars += len(cleaned)
        nonweb_total_docs += 1
        
        if len(nonweb_docs) >= SHARD_SIZE: flush_nonweb()
        if nonweb_total_docs % 5000 == 0:
            print(f'  [{nonweb_total_docs:,} non-web docs] {nonweb_total_chars:,} chars')
    
    flush_nonweb()
except Exception as e:
    flush_nonweb()
    print(f'\n⚠️ Error: {e}')

print(f'\n{"="*60}')
print(f'STEP 1 COMPLETE — PDF + ASR')
print(f'  Total streamed:    {total_streamed:,}')
print(f'  Hindi docs seen:   {lang_hits:,}')
print(f'  Type distribution: {dict(type_counts)}')
print(f'  Non-web accepted:  {nonweb_total_docs:,} ({nonweb_total_chars:,} chars)')
print(f'  Est. tokens (÷6):  ~{nonweb_total_chars // 6:,}')
print(f'{"="*60}')

---
## Step 2: Wikipedia Dump (manual, zero overlap with PDF/ASR)

In [ ]:
WIKI_DUMP = f'{WORK_DIR}/hiwiki-latest-pages-articles.xml.bz2'
if not os.path.exists(WIKI_DUMP):
    print('Downloading Hindi Wikipedia dump (~600MB, may take 10-15 min)...')
    !wget -q --show-progress -O {WIKI_DUMP} \
        'https://dumps.wikimedia.org/hiwiki/latest/hiwiki-latest-pages-articles.xml.bz2'
size_mb = os.path.getsize(WIKI_DUMP) / (1024*1024)
print(f'✅ Dump ready: {size_mb:.1f} MB')
!cp {WIKI_DUMP} {DRIVE_BASE}/raw/wiki_dump/ 2>/dev/null || true

In [ ]:
WIKI_MARKUP = [
    (re.compile(r'\{\{[^}]*\}\}'), ''),
    (re.compile(r'\[\[Category:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[श्रेणी:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[वर्ग:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[[^|\]]*\|([^\]]*)\]\]'), r'\1'),
    (re.compile(r'\[\[([^\]]*)\]\]'), r'\1'),
    (re.compile(r'\[https?://[^\s\]]*\s*([^\]]*)\]'), r'\1'),
    (re.compile(r'\[https?://[^\]]*\]'), ''),
    (re.compile(r"'{2,5}"), ''),
    (re.compile(r'<ref[^>]*>.*?</ref>', re.DOTALL), ''),
    (re.compile(r'<ref[^/>]*/>'), ''),
    (re.compile(r'<[^>]+>'), ''),
    (re.compile(r'&[a-zA-Z]+;'), ' '),
    (re.compile(r'\[\d+\]'), ''),
]
TABLE_PAT = [
    (re.compile(r'^\s*\{\|.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*\|\}.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*\|-.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*!.*$', re.MULTILINE), ''),
    (re.compile(r'^\|.*$', re.MULTILINE), ''),
]
SKIP_SEC = {'सन्दर्भ','संदर्भ','बाहरी कड़ियाँ','बाहरी कड़ी','श्रेणी','इन्हें भी देखें','यह भी देखें','स्रोत','टिप्पणी','नोट','references','external links','see also','notes'}

def clean_wiki(text):
    for p,r in TABLE_PAT: text = p.sub(r, text)
    for p,r in WIKI_MARKUP: text = p.sub(r, text)
    lines, result, skip = text.split('\n'), [], False
    for line in lines:
        s = line.strip().lower()
        if s in SKIP_SEC: skip = True; continue
        if re.match(r'^={2,}.*={2,}$', s): skip = False
        if not skip: result.append(line)
    text = '\n'.join(result)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

print('✅ Wiki cleaning ready')

In [ ]:
MANUAL_DIR = f'{WORK_DIR}/manual'
os.makedirs(MANUAL_DIR, exist_ok=True)
NS = '{http://www.mediawiki.org/xml/export-0.10/}'

man_docs, man_chars, man_count, man_shard = [], 0, 0, 0
seen, skipped_dup, rejected = 0, 0, 0

print('Processing Hindi Wikipedia dump...')
print('(PDF/ASR have zero wiki overlap → expect almost no skips)\n')

with bz2.open(WIKI_DUMP, 'rt', encoding='utf-8') as f:
    for event, elem in ET.iterparse(f, events=('end',)):
        if elem.tag != f'{NS}page': continue
        ns_el = elem.find(f'{NS}ns')
        if (ns_el is None or ns_el.text != '0') or elem.find(f'{NS}redirect') is not None:
            elem.clear(); continue
        title = (elem.find(f'{NS}title') or type('',(),{'text':''})()).text or ''
        pid = (elem.find(f'{NS}id') or type('',(),{'text':''})()).text or ''
        rev = elem.find(f'{NS}revision')
        raw = ''
        if rev is not None:
            te = rev.find(f'{NS}text')
            if te is not None and te.text: raw = te.text
        elem.clear()
        if not raw.strip(): continue
        seen += 1
        
        cleaned = clean_wiki(raw)
        if len(cleaned) < MIN_CHARS or devanagari_ratio(cleaned) < MIN_DEV_RATIO:
            rejected += 1; continue
        h = text_hash(cleaned)
        if h in all_hashes:
            skipped_dup += 1; continue
        all_hashes.add(h)
        
        man_docs.append({
            'doc_id': hashlib.md5(f'{pid}_{title}'.encode()).hexdigest()[:16],
            'title': title, 'wiki_page_id': pid,
            'source': 'Hindi Wikipedia Dump', 'source_type': 'wikipedia_dump',
            'collection_type': 'manual', 'language': LANGUAGE,
            'text': cleaned, 'clean_text_characters': len(cleaned),
            'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
            'text_sha256': h,
        })
        man_chars += len(cleaned); man_count += 1
        
        if len(man_docs) >= SHARD_SIZE:
            sf = f'{MANUAL_DIR}/wiki_shard_{man_shard:04d}.jsonl'
            with open(sf,'w',encoding='utf-8') as out:
                for d in man_docs: out.write(json.dumps(d,ensure_ascii=False)+'\n')
            print(f'  ✅ {sf}: {len(man_docs):,} docs')
            man_docs = []; man_shard += 1
        if seen % 10000 == 0:
            print(f'  Seen:{seen:,} Accept:{man_count:,} Dup:{skipped_dup:,} Reject:{rejected:,}')

if man_docs:
    sf = f'{MANUAL_DIR}/wiki_shard_{man_shard:04d}.jsonl'
    with open(sf,'w',encoding='utf-8') as out:
        for d in man_docs: out.write(json.dumps(d,ensure_ascii=False)+'\n')
    print(f'  ✅ {sf}: {len(man_docs):,} docs')

print(f'\n{"="*60}')
print(f'STEP 2 COMPLETE — WIKI DUMP (MANUAL)')
print(f'  Articles seen:    {seen:,}')
print(f'  Accepted:         {man_count:,} ({man_chars:,} chars)')
print(f'  Skipped (dup):    {skipped_dup:,}')
print(f'  Rejected:         {rejected:,}')
print(f'  Est. tokens (÷6): ~{man_chars//6:,}')
print(f'{"="*60}')

---
## Step 3: Sangraha Web + CulturaX + CC-100 (fill remaining automated)

Web content that overlaps with wiki is auto-skipped (wiki hashes in `all_hashes`).

In [ ]:
WEB_DIR = f'{WORK_DIR}/automated_web'
os.makedirs(WEB_DIR, exist_ok=True)
web_docs, web_shard_idx = [], 0
web_chars, web_count, web_skipped = 0, 0, 0

def flush_web():
    global web_docs, web_shard_idx
    if not web_docs: return
    sf = f'{WEB_DIR}/web_shard_{web_shard_idx:04d}.jsonl'
    with open(sf,'w',encoding='utf-8') as f:
        for d in web_docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
    print(f'  ✅ {sf}: {len(web_docs):,} docs')
    web_docs = []; web_shard_idx += 1

print('Streaming Sangraha Web type for Hindi...\n')
try:
    ds = load_dataset('ai4bharat/sangraha', 'verified', split='train',
                      streaming=True, trust_remote_code=True)
    for ex in ds:
        lang = ex.get('language','') or ex.get('lang','')
        if lang not in ('hin','hi','Hindi','hindi'): continue
        doc_type = (ex.get('type','') or '').strip().lower()
        if 'web' not in doc_type: continue
        text = ex.get('text','')
        if not text: continue
        cleaned = clean_web_text(text)
        if len(cleaned) < MIN_CHARS: continue
        if devanagari_ratio(cleaned) < MIN_DEV_RATIO: continue
        h = text_hash(cleaned)
        if h in all_hashes: web_skipped += 1; continue
        all_hashes.add(h)
        web_docs.append({
            'doc_id': hashlib.md5(h.encode()).hexdigest()[:16],
            'source': 'sangraha', 'source_type': 'hf_sangraha_web',
            'collection_type': 'automated', 'language': LANGUAGE,
            'text': cleaned, 'clean_text_characters': len(cleaned),
            'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
            'text_sha256': h, 'sangraha_type': doc_type,
        })
        web_chars += len(cleaned); web_count += 1
        if len(web_docs) >= SHARD_SIZE: flush_web()
        if web_count % 10000 == 0:
            print(f'  [{web_count:,} web] {web_chars:,} chars | overlap skipped: {web_skipped:,}')
    flush_web()
except Exception as e:
    flush_web()
    print(f'⚠️ Error: {e}')

print(f'\nSangraha Web: {web_count:,} docs, {web_chars:,} chars, {web_skipped:,} overlaps skipped')

In [ ]:
# CulturaX
cx_count, cx_chars = 0, 0
CX_DIR = f'{WORK_DIR}/automated_culturax'
os.makedirs(CX_DIR, exist_ok=True)
cx_docs, cx_shard = [], 0

print('Streaming CulturaX for Hindi...')
try:
    ds = load_dataset('uonlp/CulturaX', 'hi', split='train',
                      streaming=True, trust_remote_code=True)
    for ex in ds:
        text = ex.get('text','')
        if not text: continue
        cleaned = clean_web_text(text)
        if len(cleaned) < MIN_CHARS: continue
        if devanagari_ratio(cleaned) < MIN_DEV_RATIO: continue
        h = text_hash(cleaned)
        if h in all_hashes: continue
        all_hashes.add(h)
        cx_docs.append({
            'doc_id': hashlib.md5(h.encode()).hexdigest()[:16],
            'source': 'culturax', 'source_type': 'hf_culturax',
            'collection_type': 'automated', 'language': LANGUAGE,
            'text': cleaned, 'clean_text_characters': len(cleaned),
            'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
            'text_sha256': h,
        })
        cx_chars += len(cleaned); cx_count += 1
        if len(cx_docs) >= SHARD_SIZE:
            sf = f'{CX_DIR}/cx_shard_{cx_shard:04d}.jsonl'
            with open(sf,'w',encoding='utf-8') as f:
                for d in cx_docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
            print(f'  ✅ {sf}: {len(cx_docs):,} docs')
            cx_docs = []; cx_shard += 1
        if cx_count % 10000 == 0:
            print(f'  [{cx_count:,} docs] {cx_chars:,} chars')
    if cx_docs:
        sf = f'{CX_DIR}/cx_shard_{cx_shard:04d}.jsonl'
        with open(sf,'w',encoding='utf-8') as f:
            for d in cx_docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
        print(f'  ✅ {sf}: {len(cx_docs):,} docs')
except Exception as e:
    if cx_docs:
        sf = f'{CX_DIR}/cx_shard_{cx_shard:04d}.jsonl'
        with open(sf,'w',encoding='utf-8') as f:
            for d in cx_docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
    print(f'⚠️ CulturaX error: {e}')
print(f'CulturaX: {cx_count:,} docs, {cx_chars:,} chars')

In [ ]:
total_auto = nonweb_total_chars + web_chars + cx_chars
total_all_chars = man_chars + total_auto
print(f'\n{"="*60}')
print(f'COLLECTION SUMMARY')
print(f'  Manual (wiki):      {man_count:,} docs, {man_chars:,} chars')
print(f'  Auto (PDF+ASR):     {nonweb_total_docs:,} docs, {nonweb_total_chars:,} chars')
print(f'  Auto (Web):         {web_count:,} docs, {web_chars:,} chars')
print(f'  Auto (CulturaX):    {cx_count:,} docs, {cx_chars:,} chars')
print(f'  TOTAL:              {total_all_chars:,} chars (~{total_all_chars//6:,} tokens)')
print(f'  Manual %:           {man_chars/max(1,total_all_chars)*100:.1f}%')
print(f'  Unique hashes:      {len(all_hashes):,}')
print(f'{"="*60}')

---
## Step 4: Merge + Split

In [ ]:
all_docs = []
manual_total, auto_total = 0, 0
manual_char_total, auto_char_total = 0, 0
for d in [MANUAL_DIR, NONWEB_DIR, WEB_DIR, CX_DIR]:
    if not os.path.exists(d): continue
    for sf in sorted(Path(d).glob('*.jsonl')):
        with open(sf,'r',encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                doc = json.loads(line)
                all_docs.append(doc)
                c = doc.get('clean_text_characters', len(doc.get('text','')))
                if doc.get('collection_type') == 'manual':
                    manual_total += 1; manual_char_total += c
                else:
                    auto_total += 1; auto_char_total += c
tot = manual_char_total + auto_char_total
print(f'Loaded {len(all_docs):,} docs | Manual: {manual_total:,} | Auto: {auto_total:,} | Manual%: {manual_char_total/max(1,tot)*100:.1f}%')

In [ ]:
SPLITS_DIR = f'{WORK_DIR}/splits'
os.makedirs(SPLITS_DIR, exist_ok=True)
random.seed(42)
groups = {}
for doc in all_docs:
    groups.setdefault(doc.get('source_type','unknown'), []).append(doc)
train, val, test = [], [], []
for src, docs in groups.items():
    random.shuffle(docs)
    n = len(docs)
    nv = max(1, int(n*0.05)) if n > 2 else 0
    nt = max(1, int(n*0.05)) if n > 2 else 0
    train.extend(docs[:n-nv-nt])
    val.extend(docs[n-nv-nt:n-nt])
    test.extend(docs[n-nt:])
random.shuffle(train); random.shuffle(val); random.shuffle(test)

for name, docs in [('train',train),('val',val),('test',test)]:
    with open(f'{SPLITS_DIR}/{name}.jsonl','w',encoding='utf-8') as f:
        for d in docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
    with open(f'{SPLITS_DIR}/{name}.txt','w',encoding='utf-8') as f:
        for d in docs:
            t = d.get('text','')
            if t.strip(): f.write(t.strip()+'\n\n')
    chars = sum(d.get('clean_text_characters',0) for d in docs)
    mn = sum(1 for d in docs if d.get('collection_type')=='manual')
    print(f'  {name:6s}: {len(docs):>8,} docs | {chars:>14,} chars | manual: {mn:,}')
print(f'\n✅ Splits saved')

---
## Step 5: Train BPE Tokenizer

In [ ]:
import sentencepiece as spm
TDIR = f'{WORK_DIR}/tokenizer'; os.makedirs(TDIR, exist_ok=True)
MPREFIX = f'{TDIR}/hindi_bpe_32k'
TRAIN_TXT = f'{SPLITS_DIR}/train.txt'
mb = os.path.getsize(TRAIN_TXT)/(1024*1024)
print(f'Training BPE on {mb:.1f} MB... (10-30 min for Hindi)\n')

spm.SentencePieceTrainer.train(
    input=TRAIN_TXT, model_prefix=MPREFIX, vocab_size=32000,
    model_type='bpe', character_coverage=0.9995, num_threads=4,
    max_sentence_length=16384, input_sentence_size=10_000_000,
    shuffle_input_sentence=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    pad_piece='<pad>', unk_piece='<unk>', bos_piece='<bos>', eos_piece='<eos>',
    split_by_unicode_script=True, split_by_whitespace=True,
    split_digits=True, byte_fallback=True,
)
sp = spm.SentencePieceProcessor(); sp.load(f'{MPREFIX}.model')
print(f'\n✅ Tokenizer trained! Vocab: {sp.get_piece_size():,}')
t = 'भारत एक दक्षिण एशियाई देश है।'
print(f'  Test: {t} → {sp.encode(t, out_type=str)}')

---
## Step 6: Count Tokens + Upload to Drive

In [ ]:
sp = spm.SentencePieceProcessor(); sp.load(f'{MPREFIX}.model')
token_counts = {}
for sn in ['train','val','test']:
    total_tok, total_doc = 0, 0
    with open(f'{SPLITS_DIR}/{sn}.txt','r',encoding='utf-8') as f:
        buf = []
        for line in f:
            if line.strip(): buf.append(line.strip())
            elif buf:
                total_tok += len(sp.encode(' '.join(buf))); total_doc += 1; buf = []
                if total_doc % 50000 == 0: print(f'  {sn}: {total_doc:,} docs, {total_tok:,} tok')
        if buf: total_tok += len(sp.encode(' '.join(buf))); total_doc += 1
    token_counts[sn] = {'docs': total_doc, 'tokens': total_tok}
    print(f'  {sn}: {total_doc:,} docs → {total_tok:,} tokens')
TOTAL = sum(v['tokens'] for v in token_counts.values())
print(f'\n{"="*60}\nTOTAL HINDI TOKENS: {TOTAL:,}\n{"="*60}')

In [ ]:
import shutil
for subdir in ['manual','automated_nonweb','automated_web','automated_culturax']:
    src = f'{WORK_DIR}/{subdir}'
    dst = f'{DRIVE_BASE}/cleaned/{subdir}'
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        for f in Path(src).glob('*.jsonl'): shutil.copy2(str(f), dst)
        print(f'  ✅ {subdir}/')
for subdir in ['splits','tokenizer']:
    src = f'{WORK_DIR}/{subdir}'
    dst = f'{DRIVE_BASE}/{subdir}'
    if os.path.exists(src):
        for f in Path(src).iterdir(): shutil.copy2(str(f), dst)
        print(f'  ✅ {subdir}/')

summary = {
    'language': LANGUAGE,
    'manual_docs': manual_total, 'manual_chars': manual_char_total,
    'automated_docs': auto_total, 'automated_chars': auto_char_total,
    'total_docs': len(all_docs), 'total_chars': tot,
    'manual_percentage': round(manual_char_total/max(1,tot)*100, 2),
    'token_counts': token_counts, 'total_tokens': TOTAL,
    'vocab_size': sp.get_piece_size(),
}
with open(f'{DRIVE_BASE}/corpus_summary.json','w') as f:
    json.dump(summary, f, indent=2)

print(f'\n{"="*60}')
print(f'🎉 HINDI CORPUS COMPLETE!')
print(f'  Tokens:   {TOTAL:,}')
print(f'  Manual%:  {manual_char_total/max(1,tot)*100:.1f}%')
print(f'  Drive:    {DRIVE_BASE}')
print(f'{"="*60}')